# Motivation

Dieses Notebook diente zunächst nur als Testumgebung. Wir wollten zur Unterstützung von rein textuellen und fachlichen Erklärungen mehrere Clustering-Algorithmen testen und direkt visuell erkennen können, wie sich Änderungen der Parameter auf die Ergebnisse auswirken. Selbes gilt für die Normierung der Daten, welche zum Kapitel Datenaufbereitung gehört. Man findet für alle Algorithmen teils widersprüchliche Anweisungen darüber, welche Normierungsmethoden sinnvoll sind, sodass wir auch hier unsere Recherche mit praktischen Tests ergänzen wollten.

Da wir uns in unseren Erklärungen teils auf diese Tests beziehen, wollten wir unser Setup auf Nachfrage transparent zur Verfügung stellen.

In [1]:
import scipy
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.ndimage import median_filter
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
from sklearn.cluster import SpectralClustering

In [2]:
# Datensatz laden
ims_cube = scipy.io.loadmat("ims_cube.mat")['ims_cube']


## Datenaufbereitung

Die Umwandlung in eine 2D-Featurematrix ist hier ebenso notwendig.

In [3]:
# 3D zu 2D Umwandlung
print("Ursprüngliche Form des Datenwürfels:", ims_cube.shape)
spectral_matrix = ims_cube.reshape(-1, ims_cube.shape[2])
print("Form der neuen Featurematrix:", spectral_matrix.shape)


Ursprüngliche Form des Datenwürfels: (128, 128, 191)
Form der neuen Featurematrix: (16384, 191)


## Funktionen für Clustering

Hier werden alle Clustering-Funktionen definiert, die später sowohl vor, als auch nach der Normierung verwendet werden können.

**Verfügbare Funktionen:**

**Hierarchisches Clustering:**
1. `single_linkage_number(data, img_shape, n_clusters=5, n_samples_dendro=1000)` - Single Linkage mit fester Cluster-Anzahl
2. `single_linkage_distance(data, img_shape, distance_threshold=9.0, n_samples_dendro=1000)` - Single Linkage mit Distanz-Threshold
3. `complete_linkage_number(data, img_shape, n_clusters=5, n_samples_dendro=1000)` - Complete Linkage mit fester Cluster-Anzahl
4. `complete_linkage_distance(data, img_shape, distance_threshold=17.0, n_samples_dendro=1000)` - Complete Linkage mit Distanz-Threshold
5. `average_linkage_number(data, img_shape, n_clusters=5, n_samples_dendro=1000)` - Average Linkage mit fester Cluster-Anzahl
6. `average_linkage_distance(data, img_shape, distance_threshold=13.0, n_samples_dendro=1000)` - Average Linkage mit Distanz-Threshold

Bei den hierachischen Clustering-Methoden ist zur zusätzlichen Visualisierung ein Dendogramm sehr sinnvoll gewesen. Da das allerdings sehr rechenintensiv ist, wird dieses in allen Funktionen nicht mit den gesamten 16.384 Pixeln gemacht, sondern nur mit 1000 zufällig ausgewählten.

**Distanzbasiertes Clustering:**

7. `kmeans_clustering(data, img_shape, n_clusters=5)` - K-Means Clustering

**Verteilungsbasiertes Clustering:**

8. `gmm_clustering(data, img_shape, n_clusters=5, covariance_type='full')` - Gaussian Mixture Model

**Dichtebasiertes Clustering:**

9. `dbscan_clustering(data, img_shape, eps=0.5, min_samples=5)` - DBSCAN (Density-Based Clustering)

**Graph-basiertes Clustering:**

10. `spectral_clustering(data, img_shape, n_clusters=5, affinity='rbf', gamma=1.0)` - Spectral Clustering

**Parameter:**
- `data`: Featurematrix (z.B. `spectral_matrix`, `spectral_matrix_std`, `spectral_matrix_tic`, `mat_med_l2_log`)
- `img_shape`: Bildform, normalerweise `ims_cube.shape[:2]`
- `n_clusters`: Anzahl der gewünschten Cluster
- `distance_threshold`: Maximale Distanz für Cluster-Merging (nur hierarchisch)
- `n_samples_dendro`: Anzahl der Samples für Dendrogram (Standard: 1000, nur hierarchisch)
- `covariance_type`: Kovarianztyp für GMM ('full', 'tied', 'diag', 'spherical')
- `eps`: Maximale Nachbarschaftsdistanz für DBSCAN
- `min_samples`: Minimale Punktanzahl für Core-Points bei DBSCAN
- `affinity`: Ähnlichkeitsmaß für Spectral Clustering ('rbf', 'nearest_neighbors')
- `gamma`: RBF-Kernel-Parameter für Spectral Clustering

In [4]:
def single_linkage_number(data, img_shape, n_clusters=5, n_samples_dendro=1000):
    
    # Clustering durchführen
    single_linkage = AgglomerativeClustering(n_clusters=n_clusters, linkage='single')
    labels = single_linkage.fit_predict(data)
    
    # Zurück in räumliche Form bringen
    height, width = img_shape
    cluster_image = labels.reshape(height, width)
    
    # Sample für Dendrogram
    indices = np.random.choice(data.shape[0], size=min(n_samples_dendro, data.shape[0]), replace=False)
    X_sample = data[indices]
    
    # Visualisierung
    plt.figure(figsize=(12, 5))
    
    # Cluster-Bild
    plt.subplot(1, 2, 1)
    plt.imshow(cluster_image, cmap='tab10')
    plt.title(f"Single Linkage Clustering\n({n_clusters} Cluster)")
    plt.axis('off')
    plt.colorbar(label='Cluster ID')
    
    # Dendrogram
    plt.subplot(1, 2, 2)
    linkage_matrix = linkage(X_sample, method='single')
    dendrogram(linkage_matrix, no_labels=True, color_threshold=0)
    plt.title(f"Dendrogram - Single Linkage\n({n_samples_dendro} Samples)")
    plt.xlabel("Datenpunkt-Index")
    plt.ylabel("Distanz")
    
    plt.tight_layout()
    plt.show()
    
    print(f"Anzahl Cluster: {n_clusters}")
    print(f"Cluster-Größen: {np.bincount(labels)}")

In [5]:
def single_linkage_distance(data, img_shape, distance_threshold=9.0, n_samples_dendro=1000):
    
    # Clustering durchführen
    single_linkage = AgglomerativeClustering(n_clusters=None, linkage='single', distance_threshold=distance_threshold)
    labels = single_linkage.fit_predict(data)
    
    # Zurück in räumliche Form bringen
    height, width = img_shape
    cluster_image = labels.reshape(height, width)
    
    # Sample für Dendrogram
    indices = np.random.choice(data.shape[0], size=min(n_samples_dendro, data.shape[0]), replace=False)
    X_sample = data[indices]
    
    # Visualisierung
    plt.figure(figsize=(12, 5))
    
    # Cluster-Bild
    plt.subplot(1, 2, 1)
    plt.imshow(cluster_image, cmap='tab10')
    n_clusters_found = len(np.unique(labels))
    plt.title(f"Single Linkage Clustering\n({n_clusters_found} Cluster bei Threshold {distance_threshold})")
    plt.axis('off')
    plt.colorbar(label='Cluster ID')
    
    # Dendrogram mit Threshold-Linie
    plt.subplot(1, 2, 2)
    linkage_matrix = linkage(X_sample, method='single')
    dendrogram(linkage_matrix, no_labels=True, color_threshold=distance_threshold)
    plt.axhline(y=distance_threshold, color='r', linestyle='--', label=f'Threshold = {distance_threshold}')
    plt.title(f"Dendrogram - Single Linkage\n({n_samples_dendro} Samples)")
    plt.xlabel("Datenpunkt-Index")
    plt.ylabel("Distanz")
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"Distanz-Threshold: {distance_threshold}")
    print(f"Anzahl gefundener Cluster: {n_clusters_found}")
    print(f"Cluster-Größen: {np.bincount(labels)}")

In [6]:
def complete_linkage_number(data, img_shape, n_clusters=5, n_samples_dendro=1000):
    
    # Clustering durchführen
    complete_linkage = AgglomerativeClustering(n_clusters=n_clusters, linkage='complete')
    labels = complete_linkage.fit_predict(data)
    
    # Zurück in räumliche Form bringen
    height, width = img_shape
    cluster_image = labels.reshape(height, width)
    
    # Sample für Dendrogram
    indices = np.random.choice(data.shape[0], size=min(n_samples_dendro, data.shape[0]), replace=False)
    X_sample = data[indices]
    
    # Visualisierung
    plt.figure(figsize=(12, 5))
    
    # Cluster-Bild
    plt.subplot(1, 2, 1)
    plt.imshow(cluster_image, cmap='tab10')
    plt.title(f"Complete Linkage Clustering\n({n_clusters} Cluster)")
    plt.axis('off')
    plt.colorbar(label='Cluster ID')
    
    # Dendrogram
    plt.subplot(1, 2, 2)
    linkage_matrix = linkage(X_sample, method='complete')
    dendrogram(linkage_matrix, no_labels=True, color_threshold=0)
    plt.title(f"Dendrogram - Complete Linkage\n({n_samples_dendro} Samples)")
    plt.xlabel("Datenpunkt-Index")
    plt.ylabel("Distanz")
    
    plt.tight_layout()
    plt.show()
    
    print(f"Anzahl Cluster: {n_clusters}")
    print(f"Cluster-Größen: {np.bincount(labels)}")

In [7]:
def complete_linkage_distance(data, img_shape, distance_threshold=17.0, n_samples_dendro=1000):
    
    # Clustering durchführen
    complete_linkage = AgglomerativeClustering(n_clusters=None, linkage='complete', distance_threshold=distance_threshold)
    labels = complete_linkage.fit_predict(data)
    
    # Zurück in räumliche Form bringen
    height, width = img_shape
    cluster_image = labels.reshape(height, width)
    
    # Sample für Dendrogram
    indices = np.random.choice(data.shape[0], size=min(n_samples_dendro, data.shape[0]), replace=False)
    X_sample = data[indices]
    
    # Visualisierung
    plt.figure(figsize=(12, 5))
    
    # Cluster-Bild
    plt.subplot(1, 2, 1)
    plt.imshow(cluster_image, cmap='tab10')
    n_clusters_found = len(np.unique(labels))
    plt.title(f"Complete Linkage Clustering\n({n_clusters_found} Cluster bei Threshold {distance_threshold})")
    plt.axis('off')
    plt.colorbar(label='Cluster ID')
    
    # Dendrogram mit Threshold-Linie
    plt.subplot(1, 2, 2)
    linkage_matrix = linkage(X_sample, method='complete')
    dendrogram(linkage_matrix, no_labels=True, color_threshold=distance_threshold)
    plt.axhline(y=distance_threshold, color='r', linestyle='--', label=f'Threshold = {distance_threshold}')
    plt.title(f"Dendrogram - Complete Linkage\n({n_samples_dendro} Samples)")
    plt.xlabel("Datenpunkt-Index")
    plt.ylabel("Distanz")
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"Distanz-Threshold: {distance_threshold}")
    print(f"Anzahl gefundener Cluster: {n_clusters_found}")
    print(f"Cluster-Größen: {np.bincount(labels)}")

In [8]:
def average_linkage_number(data, img_shape, n_clusters=5, n_samples_dendro=1000):
    
    # Clustering durchführen
    average_linkage = AgglomerativeClustering(n_clusters=n_clusters, linkage='average')
    labels = average_linkage.fit_predict(data)
    
    # Zurück in räumliche Form bringen
    height, width = img_shape
    cluster_image = labels.reshape(height, width)
    
    # Sample für Dendrogram
    indices = np.random.choice(data.shape[0], size=min(n_samples_dendro, data.shape[0]), replace=False)
    X_sample = data[indices]
    
    # Visualisierung
    plt.figure(figsize=(12, 5))
    
    # Cluster-Bild
    plt.subplot(1, 2, 1)
    plt.imshow(cluster_image, cmap='tab10')
    plt.title(f"Average Linkage Clustering\n({n_clusters} Cluster)")
    plt.axis('off')
    plt.colorbar(label='Cluster ID')
    
    # Dendrogram
    plt.subplot(1, 2, 2)
    linkage_matrix = linkage(X_sample, method='average')
    dendrogram(linkage_matrix, no_labels=True, color_threshold=0)
    plt.title(f"Dendrogram - Average Linkage\n({n_samples_dendro} Samples)")
    plt.xlabel("Datenpunkt-Index")
    plt.ylabel("Distanz")
    
    plt.tight_layout()
    plt.show()
    
    print(f"Anzahl Cluster: {n_clusters}")
    print(f"Cluster-Größen: {np.bincount(labels)}")

In [9]:
def average_linkage_distance(data, img_shape, distance_threshold=13.0, n_samples_dendro=1000):
    
    # Clustering durchführen
    average_linkage = AgglomerativeClustering(n_clusters=None, linkage='average', distance_threshold=distance_threshold)
    labels = average_linkage.fit_predict(data)
    
    # Zurück in räumliche Form bringen
    height, width = img_shape
    cluster_image = labels.reshape(height, width)
    
    # Sample für Dendrogram
    indices = np.random.choice(data.shape[0], size=min(n_samples_dendro, data.shape[0]), replace=False)
    X_sample = data[indices]
    
    # Visualisierung
    plt.figure(figsize=(12, 5))
    
    # Cluster-Bild
    plt.subplot(1, 2, 1)
    plt.imshow(cluster_image, cmap='tab10')
    n_clusters_found = len(np.unique(labels))
    plt.title(f"Average Linkage Clustering\n({n_clusters_found} Cluster bei Threshold {distance_threshold})")
    plt.axis('off')
    plt.colorbar(label='Cluster ID')
    
    # Dendrogram mit Threshold-Linie
    plt.subplot(1, 2, 2)
    linkage_matrix = linkage(X_sample, method='average')
    dendrogram(linkage_matrix, no_labels=True, color_threshold=distance_threshold)
    plt.axhline(y=distance_threshold, color='r', linestyle='--', label=f'Threshold = {distance_threshold}')
    plt.title(f"Dendrogram - Average Linkage\n({n_samples_dendro} Samples)")
    plt.xlabel("Datenpunkt-Index")
    plt.ylabel("Distanz")
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"Distanz-Threshold: {distance_threshold}")
    print(f"Anzahl gefundener Cluster: {n_clusters_found}")
    print(f"Cluster-Größen: {np.bincount(labels)}")

In [10]:
def kmeans_clustering(data, img_shape, n_clusters=5):
    
    # Clustering durchführen
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(data)
    
    # Zurück in räumliche Form bringen
    height, width = img_shape
    cluster_image = labels.reshape(height, width)
    
    # Visualisierung
    plt.figure(figsize=(8, 6))
    plt.imshow(cluster_image, cmap='tab10')
    plt.title(f"K-Means Clustering\n({n_clusters} Cluster)")
    plt.axis('off')
    plt.colorbar(label='Cluster ID')
    plt.tight_layout()
    plt.show()
    
    print(f"Anzahl Cluster: {n_clusters}")
    print(f"Cluster-Größen: {np.bincount(labels)}")

In [11]:
def gmm_clustering(data, img_shape, n_clusters=5, covariance_type='full'):
    
    
    # Clustering durchführen
    gmm = GaussianMixture(n_components=n_clusters, covariance_type=covariance_type, random_state=42)
    gmm.fit(data)
    labels = gmm.predict(data)
    
    # Zurück in räumliche Form bringen
    height, width = img_shape
    cluster_image = labels.reshape(height, width)
    
    # Visualisierung
    plt.figure(figsize=(8, 6))
    plt.imshow(cluster_image, cmap='tab10')
    plt.title(f"GMM Clustering\n({n_clusters} Komponenten, {covariance_type} Kovarianz)")
    plt.axis('off')
    plt.colorbar(label='Cluster ID')
    plt.tight_layout()
    plt.show()
    
    print(f"Anzahl Cluster: {n_clusters}")
    print(f"Cluster-Größen: {np.bincount(labels)}")

In [12]:
def dbscan_clustering(data, img_shape, eps=0.5, min_samples=5):
    
    # Clustering durchführen
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(data) # hier enthält jeder Punkt, der keinem Cluster zugeordnet werden kann, das Label -1 (Noise)
                                      # das wird von der letzten if-Abfrage zum Zählen der Cluster und dem Anteil von Noise verwendet
    
    # Zurück in räumliche Form bringen
    height, width = img_shape
    cluster_image = labels.reshape(height, width)
    
    # Anzahl der Cluster (ohne Noise)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    # Visualisierung
    plt.figure(figsize=(8, 6))
    plt.imshow(cluster_image, cmap='tab10')
    plt.title(f"DBSCAN Clustering\n({n_clusters} Cluster, eps={eps}, min_samples={min_samples})")
    plt.axis('off')
    plt.colorbar(label='Cluster ID (-1 = Noise)')
    plt.tight_layout()
    plt.show()
    
    print(f"Anzahl Cluster: {n_clusters}")
    print(f"Anzahl Noise-Punkte: {n_noise}")
    if n_clusters > 0:
        # Nur Cluster ohne Noise zählen
        cluster_labels_only = labels[labels != -1]
        if len(cluster_labels_only) > 0:
            print(f"Cluster-Größen (ohne Noise): {np.bincount(cluster_labels_only)}")
    print(f"Anteil Noise: {n_noise / len(labels) * 100:.2f}%")

In [13]:
def spectral_clustering(data, img_shape, n_clusters=5, affinity='rbf', gamma=1.0):
    
    # Clustering durchführen
    spectral = SpectralClustering(n_clusters=n_clusters, affinity=affinity, gamma=gamma, random_state=42, n_jobs=-1)  # Nutze alle CPU-Kerne
    labels = spectral.fit_predict(data)
    
    # Zurück in räumliche Form bringen
    height, width = img_shape
    cluster_image = labels.reshape(height, width)
    
    # Visualisierung
    plt.figure(figsize=(8, 6))
    plt.imshow(cluster_image, cmap='tab10')
    plt.title(f"Spectral Clustering\n({n_clusters} Cluster, {affinity} affinity)")
    plt.axis('off')
    plt.colorbar(label='Cluster ID')
    plt.tight_layout()
    plt.show()
    
    print(f"Anzahl Cluster: {n_clusters}")
    print(f"Cluster-Größen: {np.bincount(labels)}")
    print(f"Affinity: {affinity}, Gamma: {gamma}")

### Quellen
Es wurden die in der Vorlesung betrachteten Algorithmen betrachtet. Für die Auswirkungen der Parameter wurde die Dokumentation von scikit-learn verwendet: 

https://scikit-learn.org/stable/api/sklearn.cluster.html

https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html

## Funktionsaufrufe vor Normierung

Nach einigen Tests mit unterschiedlichen Parametern sind das die Funktionsaufrufe, die am besten die charakteristischen Effekte (positiv wie negativ) der einzelnen Clustering-Algorithmen zeigen. 
Zum Beispiel kann man hier erkennen, dass Single Linkage das Problem hat, 1-Pixel Cluster zu finden, während Algorithmen wie k-means bereits ohne Normierung mit gut gewählten Parametern halbwegs brauchbare Ergebnisse liefern. Eine genaue Gegenüberstellung ausgewählter Verfahren sowie detaillierte Begründungen der Ergebnisse sind im Main-Notebook vorzufinden.

In [30]:
#Die am besten charakterisierenden Funktionsaufrufe:

#single_linkage_distance(spectral_matrix, ims_cube.shape[:2], distance_threshold=200.0)
#single_linkage_number(spectral_matrix, ims_cube.shape[:2], n_clusters=10)

#complete_linkage_number(spectral_matrix, ims_cube.shape[:2], n_clusters=11)
#complete_linkage_distance(spectral_matrix, ims_cube.shape[:2], distance_threshold=800.0)

#average_linkage_distance(spectral_matrix, ims_cube.shape[:2], distance_threshold=410.0)
#average_linkage_number(spectral_matrix, ims_cube.shape[:2], n_clusters=10)

#kmeans_clustering(spectral_matrix, ims_cube.shape[:2], n_clusters=7)

#gmm_clustering(spectral_matrix, ims_cube.shape[:2], n_clusters=10, covariance_type='full')

#dbscan_clustering(spectral_matrix, ims_cube.shape[:2], eps=125.0, min_samples=15)

#spectral_clustering(spectral_matrix, ims_cube.shape[:2], n_clusters=7, affinity='rbf', gamma=1.0)

## Normierung

Verschiedene Normierungsmethoden für die Spektraldaten.

### 1. Medianfilter pro Kanal

**Funktionsweise:** 
Für jeden Kanal wird erst einmal eine eigene 2D-Matrix analog zu der Visualisierung aus Aufgabe 1.1 erstellt, damit der Medianfilter angewendet werden kann. Anschließend werden bei jedem Pixel die angrenzenden betrachtet und der mittlere Pixel wird dann auf den Median seines unmittelbaren Umfelds gesetzt. Dadurch werden Kanten geglättet und Ausreißer entfernt (da sie keinen Einfluss auf den Median haben). 

**Beispiel:**

$$
\text{Vorher:} \quad
\begin{bmatrix}
1.0 & 2.0 & 1.5 \\
2.5 & \color{red}9.9 & \color{black}2.0 \\
1.8 & 2.2 & 1.7
\end{bmatrix}
\quad Median = 2.0 \xrightarrow{\text{Medianfilter}} \quad
\text{Nachher:} \quad
\begin{bmatrix}
1.0 & 2.0 & 1.5 \\
2.5 & \color{green}2.0 & \color{black}2.0 \\
1.8 & 2.2 & 1.7
\end{bmatrix}
\quad 
$$

**Zweck:** Entfernt Rauschen in Form von extremen Ausreißern, die nicht zu ihrem unmittelbaren Umfeld passen. Das macht das Ergebnis sauberer mit glatteren und deutlicher erkennbaren Kanten.


In [15]:
def median_filtering_per_channel(spectral_matrix, ims_cube):
    height, width = ims_cube.shape[:2]
    n_channels = spectral_matrix.shape[1]

    spectral_matrix_med = np.empty_like(spectral_matrix)
    for ch in range(n_channels):
        channel_image = spectral_matrix[:, ch].reshape(height, width)  # 2D-Bild für jeden Kanal
        channel_med = median_filter(channel_image, size=3)  # Medianfilter anwenden
        spectral_matrix_med[:, ch] = channel_med.ravel()  # Zurück in 2D-Featurematrix umwandeln
    
    return spectral_matrix_med

Quelle: https://www.geothermie.de/bibliothek/lexikon-der-geothermie/m/medianfilter

### 2. TIC-Normierung (zeilenweise)

**Funktionsweise:**
Total Ion Current (TIC) Normierung ist eine L1-Normierung, bei der jedes Spektrum (jede Zeile) durch seine Gesamtsumme geteilt wird. Dadurch wird jede Zeile auf die Summe 1 normiert. Dies gleicht Unterschiede in der Gesamtintensität zwischen verschiedenen Pixeln aus, die durch technische Faktoren (z.B. unterschiedliche Ionisierungseffizienz) entstehen können.

**Beispiel:**

Ein Spektrum mit drei m/z-Werten:
$$
\text{Vorher:} \quad [100, 200, 300] \quad \text{(Summe = 600)}
$$
$$
\xrightarrow{\text{TIC}} \quad \text{Nachher:} \quad [0.167, 0.333, 0.500] \quad \text{(Summe = 1)}
$$

**Zweck:** Normalisierung der absoluten Intensitätsunterschiede zwischen Pixeln, sodass Clustering-Algorithmen Aufschluss über die relativen Unterschiede geben, statt von Absolutwerten beeinflusst zu werden.

In [16]:
def tic_normalization(spectral_matrix):
    row_sums = spectral_matrix.sum(axis=1, keepdims=True)  # shape (n_pixels, 1)
    # Vermeide Division durch 0:
    row_sums[row_sums == 0] = 1.0
    spectral_matrix_tic = spectral_matrix / row_sums

    return spectral_matrix_tic

Quelle: https://pyopenms.readthedocs.io/en/latest/user_guide/spectrum_normalization.html#tic-normalization

### Alternativ: L2-Normierung

**Funktionsweise:**
Bei der L2-Normierung (Euklidische Normierung) wird jedes Spektrum durch seine euklidische Länge (L2-Norm) geteilt. Dadurch erhält jeder Vektor die Länge 1. Im Gegensatz zur TIC-Normierung werden hier große Werte stärker gewichtet, da sie quadratisch in die Berechnung eingehen.

**Beispiel:**

Ein Spektrum mit drei m/z-Werten:
$$
\text{Vorher:} \quad [3, 4, 0] \quad \text{(Länge = } \sqrt{3^2 + 4^2 + 0^2} = 5\text{)}
$$
$$
\xrightarrow{\text{L2}} \quad \text{Nachher:} \quad [0.6, 0.8, 0] \quad \text{(Länge = 1)}
$$

**Zweck:** Analog zur TIC-Normierung.

In [17]:
def l2_normalization(spectral_matrix):
    norms = np.linalg.norm(spectral_matrix, axis=1, keepdims=True)

    # Vermeidet Division durch 0 (fügt sehr kleinen Wert hinzu wenn Wert = 0)
    norms[norms == 0] = 1e-12

    spectral_matrix_l2 = spectral_matrix / norms

    return spectral_matrix_l2

Quellen: 
- https://mathworld.wolfram.com/L2-Norm.html
- https://numpy.org/devdocs/reference/generated/numpy.linalg.norm.html

### 3. Log-Transformation

**Funktionsweise:**
Die Log-Transformation wendet den Logarithmus auf alle Intensitätswerte an (mit `log1p`, also $\log(1+x)$, um Logarithmus mit 0 und negative Werte zu vermeiden). Dies komprimiert den Dynamikbereich und dämpft extrem hohe Peaks, sodass auch schwächere Signale sichtbar werden.

**Beispiel:**

Spektrum mit unterschiedlichen Intensitäten:
$$
\text{Vorher:} \quad [1, 10, 100, 1000]
$$
$$
\xrightarrow{\text{Log}} \quad \text{Nachher:} \quad [0.69, 2.40, 4.62, 6.91]
$$

**Zweck:** Reduzierung der Dominanz sehr hoher Peaks und bessere Erkennbarkeit von schwachen Signalen. Besonders nützlich vor Clustering-Algorithmen, die empfindlich auf Größenordnungen reagieren.

In [18]:
def log_transform(spectral_matrix):
    spectral_matrix_log = np.log1p(spectral_matrix)   # log(1 + x) damit kein log(0)

    return spectral_matrix_log

Quelle: https://medium.com/@kyawsawhtoon/log-transformation-purpose-and-interpretation-9444b4b049c9

### 4. Signal-zu-Rausch-Filterung (Feature-Selektion)

**Funktionsweise:**
Diese Methode berechnet für jeden m/z-Kanal (Spalte) das Signal-zu-Rausch-Verhältnis (SNR = Mittelwert / Standardabweichung). Kanäle mit niedrigem SNR (hohe Varianz bei niedrigem Signal) werden entfernt, da sie hauptsächlich Rauschen enthalten und wenig zur Unterscheidung von Gewebetypen beitragen.

**Beispiel:**

Drei m/z-Kanäle mit unterschiedlichem SNR:
$$
\begin{align}
\text{Kanal 1:} \quad & \text{mean} = 100, \, \text{std} = 10 \quad \Rightarrow \quad \text{SNR} = 10.0 \quad \color{green}\checkmark \\
\text{Kanal 2:} \quad & \text{mean} = 50, \, \text{std} = 45 \quad \Rightarrow \quad \text{SNR} = 1.1 \quad \color{red}\times \\
\text{Kanal 3:} \quad & \text{mean} = 200, \, \text{std} = 20 \quad \Rightarrow \quad \text{SNR} = 10.0 \quad \color{green}\checkmark
\end{align}
$$

**Zweck:** Dimensionsreduktion durch Entfernung rauschiger oder uninformativer Kanäle. Verbessert die Clustering-Qualität und reduziert Rechenzeit.

In [25]:
def snr(spectral_matrix):
    channel_std = spectral_matrix.std(axis=0)
    channel_mean = spectral_matrix.mean(axis=0)
    signal_to_noise = channel_mean / (channel_std + 1e-12)

    snr_threshold = np.percentile(signal_to_noise, 10)  # unterste 10% entfernen
    mask = (channel_std > 1e-6) & (signal_to_noise > snr_threshold)
    spectral_matrix = spectral_matrix[:, mask]

    return spectral_matrix

Quelle: https://de.wikipedia.org/wiki/Signal-Rausch-Verhältnis (Sektion: Alternative Definition)

### 5. Spaltenweise Normierung mit StandardScaler

**Funktionsweise:**
Der StandardScaler normiert jeden m/z-Kanal (Spalte) separat, sodass er Mittelwert 0 und Standardabweichung 1 hat (Z-Score-Normierung). Dies stellt sicher, dass alle Kanäle gleich stark gewichtet werden, unabhängig von ihrer ursprünglichen Intensität.

**Beispiel:**

Ein m/z-Kanal über drei Pixel (Spalte):
$$
\text{Vorher:} \quad 
\begin{bmatrix}
90 \\
100 \\
110
\end{bmatrix}
\quad \text{(mean = 100, std = 10)}
$$
$$
\xrightarrow{\text{StandardScaler}} \quad \text{Nachher:} \quad
\begin{bmatrix}
-1.0 \\
0.0 \\
1.0
\end{bmatrix}
\quad \text{(mean = 0, std = 1)}
$$

**Zweck:** Vermeidung, dass hochintensive m/z-Kanäle das Clustering dominieren. Alle Features werden auf eine vergleichbare Skala gebracht, was besonders wichtig für distanzbasierte Algorithmen (K-Means, Hierarchisches Clustering) ist.

In [20]:
def standard_scaling(spectral_matrix):
    scaler = StandardScaler()
    spectral_matrix_std = scaler.fit_transform(spectral_matrix)  # jede Spalte: mean=0, std=1

    return spectral_matrix_std

Quelle: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

### Einstellung, welche Normierungen durchgeführt werden sollen

Zu Testzwecken haben wir einige Kombinationen von Normierungen ausprobiert, um visuell direkt die Auswirkungen erkennen zu können. Die meisten dieser Kombinationen sind nicht unbedingt sinnvoll (z.B. braucht der StandardScaler annähernd normalverteilte Daten mit möglichst wenig Ausreißern (siehe Quelle)) und dienten eher als Verständnishilfe.

In [21]:
mat_med = median_filtering_per_channel(spectral_matrix, ims_cube)
mat_tic = tic_normalization(spectral_matrix)
mat_l2 = l2_normalization(spectral_matrix)
mat_log = log_transform(spectral_matrix)
mat_snr = snr(spectral_matrix)
mat_std = standard_scaling(spectral_matrix)

mat_med_tic = tic_normalization(mat_med)
mat_med_l2 = l2_normalization(mat_med)
mat_med_log = log_transform(mat_med)
mat_med_snr = snr(mat_med)
mat_med_std = standard_scaling(mat_med)

mat_med_snr_tic = tic_normalization(mat_med_snr)
mat_med_snr_tic_log = log_transform(mat_med_snr_tic)
mat_med_snr_tic_log_std = standard_scaling(mat_med_snr_tic_log)

mat_med_log_std = standard_scaling(mat_med_log)

mat_med_tic_log = log_transform(mat_med_tic)
mat_med_tic_snr = snr(mat_med_tic)
mat_med_tic_std = standard_scaling(mat_med_tic)

mat_snr_tic = tic_normalization(mat_snr)
mat_snr_tic_log = log_transform(mat_snr_tic)
mat_snr_tic_log_std = standard_scaling(mat_snr_tic_log)
mat_snr_tic_med = median_filtering_per_channel(mat_snr_tic, ims_cube)
mat_snr_tic_med_log = log_transform(mat_snr_tic_med)
mat_snr_tic_med_log_std = standard_scaling(mat_snr_tic_med_log)

mat_med_l2_log = log_transform(mat_med_l2)
mat_med_l2_snr = snr(mat_med_l2)
mat_med_l2_std = standard_scaling(mat_med_l2)

mat_med_tic_log_snr = snr(mat_med_tic_log)
mat_med_tic_log_std = standard_scaling(mat_med_tic_log)

mat_med_l2_log_snr = snr(mat_med_l2_log)
mat_med_l2_log_std = standard_scaling(mat_med_l2_log)

mat_tic_log= log_transform(mat_tic)
mat_tic_snr = snr(mat_tic)
mat_tic_std = standard_scaling(mat_tic)

mat_l2_log= log_transform(mat_l2)
mat_l2_snr = snr(mat_l2)    
mat_l2_std = standard_scaling(mat_l2)

mat_l2_log_snr = snr(mat_l2_log)

mat_tic_log_std = standard_scaling(mat_tic_log)

mat_std_l2 = l2_normalization(mat_std)
mat_std_tic = tic_normalization(mat_std)

mat_all_tic = standard_scaling(mat_med_tic_log_snr)
mat_all_l2 = standard_scaling(mat_med_l2_log_snr)

mat_l2_log_snr_std = standard_scaling(mat_l2_log_snr)

## Funktionsaufrufe nach Normierung

Hier die am besten charakterisierenden Funktionsaufrufe, genau wie vor der Normierung. Hier kommen allerdings noch die verschiedenen Möglichkeiten hinsichtlich der Auswahl der Normierungsmethoden hinzu. 

In [33]:
# Die am besten charakterisierenden Funktionsaufrufe:

#single_linkage_number(mat_snr_tic_med_log_std, ims_cube.shape[:2], n_clusters=10) # bestmögliches Resultat von Single Linkage mit optimalen Normierungen

#complete_linkage_number(mat_med_l2_log_snr, ims_cube.shape[:2], n_clusters=11)

#average_linkage_number(mat_med_l2_log, ims_cube.shape[:2], n_clusters=12)

#kmeans_clustering(mat_all_tic, ims_cube.shape[:2], n_clusters=7)

#gmm_clustering(mat_med_snr_tic_log_std, ims_cube.shape[:2], n_clusters=7, covariance_type='tied')

#dbscan_clustering(mat_tic, ims_cube.shape[:2], eps=0.029, min_samples=5)

#spectral_clustering(mat_med_snr_tic_log_std, ims_cube.shape[:2], n_clusters=7, affinity='rbf', gamma=1.0)

Anmerkung: Alle Quellen in diesem Notebook wurden zuletzt am 04.11.2025 aufgerufen.